In [1]:
# Drive Mounting and Dataset Loading

from google.colab import drive
drive.mount("/content/drive")

import torch
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Major_Project/Tuberculosis")
OUTPUT_DIR = BASE / "outputs"

# Load Fixed Dataset Splits
train_df = pd.read_csv(OUTPUT_DIR / "tb_train.csv")
val_df = pd.read_csv(OUTPUT_DIR / "tb_val.csv")
test_df = pd.read_csv(OUTPUT_DIR / "tb_test.csv")

# Locate Image Directory
IMAGE_DIR = Path(train_df["image_path"].iloc[0]).parent

# GPU Configuration
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Environment Verification
print("Device:", device)
print("PyTorch version:", torch.__version__)

print("\nTraining images:", len(train_df))
print("Validation images:", len(val_df))
print("Testing images:", len(test_df))

print("\nImage directory exists:", IMAGE_DIR.exists())
print("Sample image exists:",
      Path(train_df["image_path"].iloc[0]).exists())

print("Preprocessing file exists:",
      (OUTPUT_DIR / "tb_preprocessing.py").exists())


Mounted at /content/drive
Device: cuda
PyTorch version: 2.14.1+cu130

Training images: 463
Validation images: 99
Testing images: 100

Image directory exists: True
Sample image exists: True
Preprocessing file exists: True


In [2]:
# TB Dataset and DataLoader

import sys
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Load Saved Preprocessing Function
sys.path.insert(0, str(OUTPUT_DIR))
from tb_preprocessing import preprocess_tb_image

# ImageNet Normalization
image_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Custom TB Dataset
class TBDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image = preprocess_tb_image(row["image_path"])
        image = image_transform(image)

        label = int(row["label"])

        return image, label

# Create Datasets
train_dataset = TBDataset(train_df)
val_dataset = TBDataset(val_df)
test_dataset = TBDataset(test_df)

# Create DataLoaders
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

# Verify DataLoader
images, labels = next(iter(train_loader))

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)
print("Labels:", labels.tolist())
print("Pixel values are finite:", torch.isfinite(images).all().item())


Training dataset: 463
Validation dataset: 99
Test dataset: 100
Batch image shape: torch.Size([16, 3, 224, 224])
Batch label shape: torch.Size([16])
Labels: [0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0]
Pixel values are finite: True


In [3]:
# DenseNet-121 Model Initialization

import torch
import torch.nn as nn
from torchvision.models import densenet121, DenseNet121_Weights

# Load Pretrained CNN
weights = DenseNet121_Weights.DEFAULT
cnn_model = densenet121(weights=weights)

# Configure Binary Classification
num_features = cnn_model.classifier.in_features
cnn_model.classifier = nn.Linear(num_features, 2)

# Move Model to Device
cnn_model = cnn_model.to(device)

# Verify Model
print("Model: DenseNet-121")
print("Output classes:", cnn_model.classifier.out_features)
print("Device:", next(cnn_model.parameters()).device)

total_params = sum(p.numel() for p in cnn_model.parameters())
trainable_params = sum(
    p.numel() for p in cnn_model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Verify Output Shape
cnn_model.eval()

with torch.no_grad():
    sample_batch = images[:2].to(device)
    sample_output = cnn_model(sample_batch)

print("Sample output shape:", sample_output.shape)


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 162MB/s]


Model: DenseNet-121
Output classes: 2
Device: cuda:0
Total parameters: 6,955,906
Trainable parameters: 6,955,906
Sample output shape: torch.Size([2, 2])


In [4]:
# CNN Training Configuration

import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Training Hyperparameters
NUM_CLASSES = 2
EPOCHS = 10
LEARNING_RATE = 0.0001

# Loss Function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.AdamW(
    cnn_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.0001
)

# Learning Rate Scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

# Verify Configuration
print("Model: DenseNet-121")
print("Loss Function: CrossEntropyLoss")
print("Optimizer: AdamW")
print("Learning Rate:", LEARNING_RATE)
print("Epochs:", EPOCHS)
print("Batch Size:", BATCH_SIZE)
print("Number of Classes:", NUM_CLASSES)
print("Device:", device)


Model: DenseNet-121
Loss Function: CrossEntropyLoss
Optimizer: AdamW
Learning Rate: 0.0001
Epochs: 10
Batch Size: 16
Number of Classes: 2
Device: cuda


In [5]:
# DenseNet-121 Training and Validation

from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import torch

# Model Checkpoint Directory
MODEL_DIR = OUTPUT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Training History
cnn_history = []
best_val_auc = -1.0
best_epoch = 0

# Training Loop
for epoch in range(EPOCHS):

    cnn_model.train()
    train_loss_sum = 0.0

    for images, labels in tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS} - Training"
    ):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = cnn_model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * images.size(0)

    train_loss = train_loss_sum / len(train_dataset)

    # Validation
    cnn_model.eval()
    val_loss_sum = 0.0
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in tqdm(
            val_loader,
            desc=f"Epoch {epoch + 1}/{EPOCHS} - Validation"
        ):
            images = images.to(device)
            labels = labels.to(device)

            outputs = cnn_model(images)
            loss = criterion(outputs, labels)

            probabilities = torch.softmax(outputs, dim=1)[:, 1]

            val_loss_sum += loss.item() * images.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probabilities.cpu().numpy())

    val_loss = val_loss_sum / len(val_dataset)

    # Binary Validation AUROC
    val_auc = roc_auc_score(
        np.array(all_labels),
        np.array(all_probs)
    )

    # Save Best Checkpoint
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_epoch = epoch + 1

        torch.save(
            cnn_model.state_dict(),
            MODEL_DIR / "tb_densenet121_best.pth"
        )

    # Update Learning Rate
    scheduler.step()

    # Record Metrics
    cnn_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_auc": val_auc
    })

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val AUROC: {val_auc:.4f}"
    )

    # Save History After Every Epoch
    pd.DataFrame(cnn_history).to_csv(
        OUTPUT_DIR / "tb_densenet121_training_history.csv",
        index=False
    )

# Training Summary
print("\nDenseNet-121 training completed.")
print("Best epoch:", best_epoch)
print(f"Best validation AUROC: {best_val_auc:.4f}")
print("Best checkpoint:", MODEL_DIR / "tb_densenet121_best.pth")


Epoch 1/10 - Training:   0%|          | 0/29 [00:00<?, ?it/s]

Epoch 1/10 - Validation:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch 1/10 | Train Loss: 0.4532 | Val Loss: 0.4340 | Val AUROC: 0.8967


Epoch 2/10 - Training:   0%|          | 0/29 [00:00<?, ?it/s]

Epoch 2/10 - Validation:   0%|          | 0/7 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b1361f98a40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b1361f98a40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

... (28 outputs left out)

Epoch 10/10 - Training:   0%|          | 0/29 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b1361f98a40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b1361f98a40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, 

Epoch 10/10 - Validation:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch 10/10 | Train Loss: 0.0067 | Val Loss: 0.3685 | Val AUROC: 0.9416

DenseNet-121 training completed.
Best epoch: 4
Best validation AUROC: 0.9482
Best checkpoint: /content/drive/MyDrive/Major_Project/Tuberculosis/outputs/models/tb_densenet121_best.pth


In [6]:
# DenseNet Validation Metrics

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
import numpy as np
import pandas as pd
import torch

# Load Best Checkpoint
checkpoint_path = MODEL_DIR / "tb_densenet121_best.pth"
cnn_model.load_state_dict(
    torch.load(checkpoint_path, map_location=device, weights_only=True)
)
cnn_model.eval()

# Validation Predictions
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)

        outputs = cnn_model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1]

        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

y_true = np.array(all_labels)
y_prob = np.array(all_probs)
y_pred = (y_prob >= 0.5).astype(int)

# Calculate Metrics
metrics = {
    "model": "DenseNet-121",
    "auroc": roc_auc_score(y_true, y_prob),
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, zero_division=0),
    "recall": recall_score(y_true, y_pred, zero_division=0),
    "f1_score": f1_score(y_true, y_pred, zero_division=0),
    "brier_loss": np.mean((y_prob - y_true) ** 2)
}

# Save Results
metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv(
    OUTPUT_DIR / "tb_densenet121_validation_metrics.csv",
    index=False
)

print("DenseNet-121 Validation Metrics:")
print(metrics_df.to_string(index=False))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))


DenseNet-121 Validation Metrics:
       model    auroc  accuracy  precision  recall  f1_score  brier_loss
DenseNet-121 0.948163  0.878788   0.913043    0.84     0.875     0.09575

Confusion Matrix:
[[45  4]
 [ 8 42]]


In [7]:
# ViT-B/16 Model Initialization

import torch
import torch.nn as nn
from torchvision.models import vit_b_16, ViT_B_16_Weights

# Load ImageNet-Pretrained ViT
vit_weights = ViT_B_16_Weights.DEFAULT
vit_model = vit_b_16(weights=vit_weights)

# Replace Classifier for TB Classification
num_features = vit_model.heads.head.in_features
vit_model.heads.head = nn.Linear(num_features, 2)

# Move Model to GPU
vit_model = vit_model.to(device)

# Verify Model
print("Model: ViT-B/16")
print("Output classes:", vit_model.heads.head.out_features)
print("Device:", next(vit_model.parameters()).device)

total_params = sum(p.numel() for p in vit_model.parameters())
trainable_params = sum(
    p.numel() for p in vit_model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:03<00:00, 111MB/s]


Model: ViT-B/16
Output classes: 2
Device: cuda:0
Total parameters: 85,800,194
Trainable parameters: 85,800,194


In [8]:
# ViT Training Configuration

import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Training Hyperparameters
VIT_EPOCHS = 10
VIT_LEARNING_RATE = 0.0001
VIT_BATCH_SIZE = 4
VIT_ACCUMULATION_STEPS = 4

# ViT DataLoaders
vit_train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=VIT_BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

vit_val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=VIT_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

# Loss Function
vit_criterion = nn.CrossEntropyLoss()

# Optimizer
vit_optimizer = optim.AdamW(
    vit_model.parameters(),
    lr=VIT_LEARNING_RATE,
    weight_decay=0.0001
)

# Learning Rate Scheduler
vit_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    vit_optimizer,
    T_max=VIT_EPOCHS
)

# Mixed Precision
use_amp = device.type == "cuda"

vit_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp
)

# Verify Configuration
print("Model: ViT-B/16")
print("Epochs:", VIT_EPOCHS)
print("Learning Rate:", VIT_LEARNING_RATE)
print("Physical Batch Size:", VIT_BATCH_SIZE)
print("Gradient Accumulation Steps:", VIT_ACCUMULATION_STEPS)
print("Effective Batch Size:",
      VIT_BATCH_SIZE * VIT_ACCUMULATION_STEPS)
print("Mixed Precision Enabled:", use_amp)
print("Device:", device)


Model: ViT-B/16
Epochs: 10
Learning Rate: 0.0001
Physical Batch Size: 4
Gradient Accumulation Steps: 4
Effective Batch Size: 16
Mixed Precision Enabled: True
Device: cuda


In [9]:
# ViT-B/16 Training and Validation

from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import torch

# Checkpoint Directory
MODEL_DIR = OUTPUT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Training History
vit_history = []
best_vit_auc = -1.0
best_vit_epoch = 0

# Training Loop
for epoch in range(VIT_EPOCHS):

    vit_model.train()
    vit_optimizer.zero_grad()
    train_loss_sum = 0.0

    for batch_idx, (images, labels) in enumerate(
        tqdm(
            vit_train_loader,
            desc=f"Epoch {epoch + 1}/{VIT_EPOCHS} - Training"
        )
    ):
        images = images.to(device)
        labels = labels.to(device)

        # Mixed Precision Forward Pass
        with torch.autocast(
            device_type="cuda",
            enabled=use_amp
        ):
            outputs = vit_model(images)
            loss = vit_criterion(outputs, labels)

        # Gradient Accumulation
        batch_count = len(vit_train_loader)
        group_start = (
            batch_idx // VIT_ACCUMULATION_STEPS
        ) * VIT_ACCUMULATION_STEPS
        group_size = min(
            VIT_ACCUMULATION_STEPS,
            batch_count - group_start
        )

        vit_scaler.scale(loss / group_size).backward()

        if (
            (batch_idx + 1) % VIT_ACCUMULATION_STEPS == 0
            or (batch_idx + 1) == batch_count
        ):
            vit_scaler.step(vit_optimizer)
            vit_scaler.update()
            vit_optimizer.zero_grad()

        train_loss_sum += loss.item() * images.size(0)

    train_loss = train_loss_sum / len(train_dataset)

    # Validation
    vit_model.eval()
    val_loss_sum = 0.0
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in tqdm(
            vit_val_loader,
            desc=f"Epoch {epoch + 1}/{VIT_EPOCHS} - Validation"
        ):
            images = images.to(device)
            labels = labels.to(device)

            with torch.autocast(
                device_type="cuda",
                enabled=use_amp
            ):
                outputs = vit_model(images)
                loss = vit_criterion(outputs, labels)

            probs = torch.softmax(
                outputs.float(), dim=1
            )[:, 1]

            val_loss_sum += loss.item() * images.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    val_loss = val_loss_sum / len(val_dataset)

    # Binary Validation AUROC
    val_auc = roc_auc_score(
        np.array(all_labels),
        np.array(all_probs)
    )

    # Save Best ViT Checkpoint
    if val_auc > best_vit_auc:
        best_vit_auc = val_auc
        best_vit_epoch = epoch + 1

        torch.save(
            vit_model.state_dict(),
            MODEL_DIR / "tb_vit_b16_best.pth"
        )

    # Update Learning Rate
    vit_scheduler.step()

    # Record Results
    vit_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_auc": val_auc
    })

    print(
        f"Epoch {epoch + 1}/{VIT_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val AUROC: {val_auc:.4f}"
    )

    # Save History
    pd.DataFrame(vit_history).to_csv(
        OUTPUT_DIR / "tb_vit_b16_training_history.csv",
        index=False
    )

# Training Summary
print("\nViT-B/16 training completed.")
print("Best epoch:", best_vit_epoch)
print(f"Best validation AUROC: {best_vit_auc:.4f}")
print("Best checkpoint:", MODEL_DIR / "tb_vit_b16_best.pth")


Epoch 1/10 - Training:   0%|          | 0/116 [00:00<?, ?it/s]

Epoch 1/10 - Validation:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch 1/10 | Train Loss: 0.6349 | Val Loss: 0.4494 | Val AUROC: 0.8776


Epoch 2/10 - Training:   0%|          | 0/116 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b1361f98a40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b1361f98a40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Epoch 2/10 - Validation:   0%|          | 0/25 [00:00<?, ?it/s]

... (31 outputs left out)

Epoch 10/10 - Training:   0%|          | 0/116 [00:00<?, ?it/s]

Epoch 10/10 - Validation:   0%|          | 0/25 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b1361f98a40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process


Epoch 10/10 | Train Loss: 0.0021 | Val Loss: 0.7142 | Val AUROC: 0.8896

ViT-B/16 training completed.
Best epoch: 3
Best validation AUROC: 0.9057
Best checkpoint: /content/drive/MyDrive/Major_Project/Tuberculosis/outputs/models/tb_vit_b16_best.pth


In [10]:
# ViT Validation Metrics

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
import numpy as np
import pandas as pd
import torch

# Load Best ViT Checkpoint
vit_model.load_state_dict(
    torch.load(
        MODEL_DIR / "tb_vit_b16_best.pth",
        map_location=device,
        weights_only=True
    )
)

vit_model.eval()

# Validation Predictions
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in vit_val_loader:
        images = images.to(device)

        with torch.autocast(
            device_type="cuda",
            enabled=use_amp
        ):
            outputs = vit_model(images)

        probs = torch.softmax(
            outputs.float(), dim=1
        )[:, 1]

        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

y_true = np.array(all_labels)
y_prob = np.array(all_probs)
y_pred = (y_prob >= 0.5).astype(int)

# Calculate Metrics
vit_metrics = {
    "model": "ViT-B/16",
    "auroc": roc_auc_score(y_true, y_prob),
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(
        y_true, y_pred, zero_division=0
    ),
    "recall": recall_score(
        y_true, y_pred, zero_division=0
    ),
    "f1_score": f1_score(
        y_true, y_pred, zero_division=0
    ),
    "brier_loss": np.mean((y_prob - y_true) ** 2)
}

# Save Results
vit_metrics_df = pd.DataFrame([vit_metrics])

vit_metrics_df.to_csv(
    OUTPUT_DIR / "tb_vit_b16_validation_metrics.csv",
    index=False
)

print("ViT-B/16 Validation Metrics:")
print(vit_metrics_df.to_string(index=False))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))


ViT-B/16 Validation Metrics:
   model    auroc  accuracy  precision  recall  f1_score  brier_loss
ViT-B/16 0.905714  0.828283   0.866667    0.78  0.821053    0.121347

Confusion Matrix:
[[43  6]
 [11 39]]


In [11]:
# TB Baseline Model Selection

import pandas as pd
import json

# Load Saved Validation Metrics
cnn_results = pd.read_csv(
    OUTPUT_DIR / "tb_densenet121_validation_metrics.csv"
)

vit_results = pd.read_csv(
    OUTPUT_DIR / "tb_vit_b16_validation_metrics.csv"
)

# Compare Candidate Models
comparison_df = pd.concat(
    [cnn_results, vit_results],
    ignore_index=True
)

comparison_df = comparison_df.sort_values(
    by="auroc",
    ascending=False
).reset_index(drop=True)

# Select Baseline Using Validation AUROC
winner = comparison_df.iloc[0]
baseline_model = winner["model"]

checkpoint_names = {
    "DenseNet-121": "tb_densenet121_best.pth",
    "ViT-B/16": "tb_vit_b16_best.pth"
}

baseline_checkpoint = (
    OUTPUT_DIR / "models" / checkpoint_names[baseline_model]
)

# Save Comparison
comparison_df.to_csv(
    OUTPUT_DIR / "tb_cnn_vs_vit_comparison.csv",
    index=False
)

# Save Baseline Information
baseline_info = {
    "selected_model": baseline_model,
    "selection_metric": "validation_auroc",
    "validation_auroc": float(winner["auroc"]),
    "checkpoint": str(baseline_checkpoint),
    "dataset": "Shenzhen Chest X-rays",
    "task": "Binary TB-related abnormality classification"
}

with open(OUTPUT_DIR / "tb_baseline_selection.json", "w") as f:
    json.dump(baseline_info, f, indent=4)

# Display Results
print("CNN vs ViT Comparison:")
print(comparison_df.to_string(index=False))

print("\nSelected TB Baseline:", baseline_model)
print("Baseline Validation AUROC:", round(float(winner["auroc"]), 4))
print("Checkpoint exists:", baseline_checkpoint.exists())
print("Baseline selection saved successfully.")


CNN vs ViT Comparison:
       model    auroc  accuracy  precision  recall  f1_score  brier_loss
DenseNet-121 0.948163  0.878788   0.913043    0.84  0.875000    0.095750
    ViT-B/16 0.905714  0.828283   0.866667    0.78  0.821053    0.121347

Selected TB Baseline: DenseNet-121
Baseline Validation AUROC: 0.9482
Checkpoint exists: True
Baseline selection saved successfully.
